#Part A

In [40]:
import numpy as np
import matplotlib.pyplot as plt

T = 20000
MC_RUNS = 20
N_explore = int(T**(2/3))
k = 2
eta = ((k * np.log(k)) / T) ** (1/3)
epsilon = ((k * np.log(k)) / T) ** (1/3)

"""
#-------------- MW wins UCB--------------------

# Arm 1 parameters
p1_L = 0.1
p1_H = 0.99
T1_L = 13000
T1_H =  7000
initial_state_1 = "H"

# Arm 2 parameters
p2_L = 0.5
p2_H = 0.8
T2_L = 13000
T2_H = 7000
initial_state_2 = "H"
"""
# -------------MW wins EtE--------------------
# Arm 1 parameters
p1_L = 0.0
p1_H = 0.99
T1_L =  18000
T1_H =  2000
initial_state_1 = "H"

# Arm 2 parameters
p2_L = 0.6
p2_H = 0.8
T2_L =  18000
T2_H =  2000
initial_state_2 = "H"

"""
#----------Sanity check--------------------
# Arm 1 parameters
p1_L = 0.8
p1_H = 0.8
T1_L = 700
T1_H = 700
initial_state_1 = "H"

# Arm 2 parameters
p2_L = 0.3
p2_H = 0.3
T2_L = 700
T2_H = 700
initial_state_2 = "L"
"""


def build_pattern(pL, pH, TL, TH, initial_state):
    pattern = []
    if initial_state == "L":
        pattern = [pL] * TL + [pH] * TH
        return pattern
    else:
        pattern = [pH] * TH + [pL] * TL
        return pattern

def optimal_expected_cum_reward():
    pattern1 = build_pattern(p1_L, p1_H, T1_L, T1_H, initial_state_1)
    pattern2 = build_pattern(p2_L, p2_H, T2_L, T2_H, initial_state_2)

    period1 = len(pattern1)
    period2 = len(pattern2)

    opt_cum = []
    cumulative = 0.0

    for t in range(1, T + 1):
        p1_t = pattern1[(t - 1) % period1]
        p2_t = pattern2[(t - 1) % period2]

        best_prob = max(p1_t, p2_t)
        cumulative += best_prob
        opt_cum.append(cumulative)

    return opt_cum

def ucb(mc_seed):
    rng = np.random.default_rng(mc_seed)

    N = [0, 0]
    reward_sums = [0.0, 0.0]
    mu_hat = [0.0, 0.0]

    rewards = []
    cum_rewards = []
    chosen_arms = []

    cumulative_reward = 0.0

    pattern1 = build_pattern(p1_L, p1_H, T1_L, T1_H, initial_state_1)
    pattern2 = build_pattern(p2_L, p2_H, T2_L, T2_H, initial_state_2)
    period1 = len(pattern1)
    period2 = len(pattern2)

    for t in range(1, T + 1):
        if N[0] == 0:
            arm = 0
        elif N[1] == 0:
            arm = 1
        else:
            ucb_0 = mu_hat[0] + np.sqrt((2 * np.log(T)) / N[0])
            ucb_1 = mu_hat[1] + np.sqrt((2 * np.log(T)) / N[1])

            if ucb_0 >= ucb_1:
                arm = 0
            else:
                arm = 1

        chosen_arms.append(arm)

        pos1 = (t - 1) % period1
        p1_t = pattern1[pos1]

        pos2 = (t - 1) % period2
        p2_t = pattern2[pos2]

        prob_of_arms = [p1_t, p2_t]
        p = prob_of_arms[arm]

        reward = rng.binomial(1, p)
        rewards.append(reward)

        cumulative_reward += reward
        cum_rewards.append(cumulative_reward)

        N[arm] += 1
        reward_sums[arm] += reward
        mu_hat[arm] = reward_sums[arm] / N[arm]

    return rewards, cum_rewards, reward_sums, mu_hat, chosen_arms

def ete(mc_seed, N_explore):
    rng = np.random.default_rng(mc_seed)

    N = [0, 0]
    reward_sums = [0.0, 0.0]
    mu_hat = [0.0, 0.0]

    rewards = []
    cum_rewards = []
    chosen_arms = []

    cumulative_reward = 0.0

    pattern1 = build_pattern(p1_L, p1_H, T1_L, T1_H, initial_state_1)
    pattern2 = build_pattern(p2_L, p2_H, T2_L, T2_H, initial_state_2)
    period1 = len(pattern1)
    period2 = len(pattern2)

    commit_done = False

    for t in range(1, T + 1):

        if t <= 2 * N_explore:
            arm = (t - 1) % 2
        else: # we need to enter it only once after end of explore phase , to do that we use the flag "commit_done = True/False"
            if not commit_done:
                if mu_hat[0] >= mu_hat[1]:
                    committed_arm = 0
                else:
                    committed_arm = 1
                commit_done = True

            arm = committed_arm

        chosen_arms.append(arm)

        pos1 = (t - 1) % period1
        p1_t = pattern1[pos1]

        pos2 = (t - 1) % period2
        p2_t = pattern2[pos2]

        prob_of_arms = [p1_t, p2_t]
        p = prob_of_arms[arm]

        reward = rng.binomial(1, p)
        rewards.append(reward)

        cumulative_reward += reward
        cum_rewards.append(cumulative_reward)

        N[arm] += 1
        reward_sums[arm] += reward
        mu_hat[arm] = reward_sums[arm] / N[arm]

    return rewards, cum_rewards, reward_sums, mu_hat, chosen_arms

def mw_bandit(mc_seed, eta, epsilon):
    rng = np.random.default_rng(mc_seed)
    weights = np.ones(k)

    rewards = []
    cum_rewards = []
    chosen_arms = []
    cumulative_reward = 0.0

    pattern1 = build_pattern(p1_L, p1_H, T1_L, T1_H, initial_state_1)
    pattern2 = build_pattern(p2_L, p2_H, T2_L, T2_H, initial_state_2)
    period1 = len(pattern1)
    period2 = len(pattern2)

    for t in range(1, T + 1):
        p = weights / np.sum(weights)
        q = (1 - epsilon) * p + epsilon / k

        arm = rng.choice(k, p=q)
        chosen_arms.append(arm)

        pos1 = (t - 1) % period1
        p1_t = pattern1[pos1]

        pos2 = (t - 1) % period2
        p2_t = pattern2[pos2]

        prob_of_arms = [p1_t, p2_t]
        reward_prob = prob_of_arms[arm]

        reward = rng.binomial(1, reward_prob)
        rewards.append(reward)

        cumulative_reward += reward
        cum_rewards.append(cumulative_reward)

        li = 1 - reward
        li_hat = li / q[arm]
        weights[arm] = weights[arm] * ((1 - eta) ** li_hat)

    return rewards, cum_rewards, chosen_arms

def plot_comparison(mean_cum_rewards_ucb, mean_cum_rewards_ete, mean_cum_rewards_mw, optimal_cum_reward):
    t_axis = range(1, T + 1)

    plt.figure()
    plt.plot(t_axis, mean_cum_rewards_ucb, label="UCB")
    plt.plot(t_axis, mean_cum_rewards_mw, label="MW")
    plt.plot(t_axis, mean_cum_rewards_ete, label="EtE")
    plt.plot(t_axis, optimal_cum_reward, label="Optimal", linestyle="--")

    plt.xlabel("t")
    plt.ylabel("Cumulative reward")
    plt.title("Cumulative Reward Comparison")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig("Comparison_Cumulative_Reward.png", dpi=300, bbox_inches='tight')
    plt.close()

def plot_arm_selection_per_phase(all_chosen_arms_alg1, all_chosen_arms_alg2,
                                 label1, label2, filename):
    all_chosen_arms_alg1 = np.array(all_chosen_arms_alg1)
    all_chosen_arms_alg2 = np.array(all_chosen_arms_alg2)

    alg1_arm1 = []
    alg2_arm1 = []

    alg1_arm2 = []
    alg2_arm2 = []

    phase_names = []

    start = 0
    phase_idx = 0

    while start < T:
        if phase_idx % 2 == 0:
            phase_len = T1_H
            phase_names.append("H")
        else:
            phase_len = T1_L
            phase_names.append("L")

        end = min(start + phase_len, T)

        alg1_frac_arm1 = np.mean(all_chosen_arms_alg1[:, start:end] == 0)
        alg2_frac_arm1 = np.mean(all_chosen_arms_alg2[:, start:end] == 0)

        alg1_arm1.append(alg1_frac_arm1)
        alg2_arm1.append(alg2_frac_arm1)

        alg1_arm2.append(1 - alg1_frac_arm1)
        alg2_arm2.append(1 - alg2_frac_arm1)

        start = end
        phase_idx += 1

    x = np.arange(len(phase_names))
    width = 0.35

    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.bar(x - width/2, alg1_arm1, width=width, label=label1)
    plt.bar(x + width/2, alg2_arm1, width=width, label=label2)
    plt.xlabel("Phase index")
    plt.ylabel("Mean selection frequency")
    plt.title("Arm 1 Selection Frequency per Phase")
    plt.xticks(x, phase_names)
    plt.ylim(0, 1)
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.bar(x - width/2, alg1_arm2, width=width, label=label1)
    plt.bar(x + width/2, alg2_arm2, width=width, label=label2)
    plt.xlabel("Phase index")
    plt.ylabel("Mean selection frequency")
    plt.title("Arm 2 Selection Frequency per Phase")
    plt.xticks(x, phase_names)
    plt.ylim(0, 1)
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

if __name__ == "__main__":

    # the following are lists of length (mc x T) where we keep the cumulative reward of each algorithm for each r MC run
    # so this looks like : [
    #                        [run0 cumulative rewards over T],
    #                        [run1 cumulative rewards over T],
    #                        [run2 cumulative rewards over T],
    #                          ...
    #                       ]

    all_cum_rewards_ucb = []
    all_cum_rewards_ete = []
    all_cum_rewards_mw = []
    # same as above but here we keep the arms played for each run
    all_chosen_arms_ucb = []
    all_chosen_arms_ete = []
    all_chosen_arms_mw = []

    for mc in range(MC_RUNS):
        rewards_ucb, cum_rewards_ucb, reward_sums_ucb, mu_hat_ucb, chosen_arms_ucb = ucb(mc)
        all_cum_rewards_ucb.append(cum_rewards_ucb)
        all_chosen_arms_ucb.append(chosen_arms_ucb)

        rewards_ete, cum_rewards_ete, reward_sums_ete, mu_hat_ete, chosen_arms_ete = ete(mc, N_explore)
        all_cum_rewards_ete.append(cum_rewards_ete)
        all_chosen_arms_ete.append(chosen_arms_ete)

        rewards_mw, cum_rewards_mw, chosen_arms_mw = mw_bandit(mc, eta, epsilon)
        all_cum_rewards_mw.append(cum_rewards_mw)
        all_chosen_arms_mw.append(chosen_arms_mw)
    # now the following lists will keep the average over runs and are of lenght T
    mean_cum_rewards_ucb = []
    mean_cum_rewards_ete = []
    mean_cum_rewards_mw = []
    # for specific t, we average accross all MC runs by firat summing the cum reward of each run and dividing by number of MC runs
    for t in range(T):
        mean_reward_t_ucb = sum(all_cum_rewards_ucb[mc][t] for mc in range(MC_RUNS)) / MC_RUNS
        mean_reward_t_ete = sum(all_cum_rewards_ete[mc][t] for mc in range(MC_RUNS)) / MC_RUNS
        mean_reward_t_mw = sum(all_cum_rewards_mw[mc][t] for mc in range(MC_RUNS)) / MC_RUNS

        mean_cum_rewards_ucb.append(mean_reward_t_ucb)
        mean_cum_rewards_ete.append(mean_reward_t_ete)
        mean_cum_rewards_mw.append(mean_reward_t_mw)
    # since now we need the total reward collected by the end of the horizon we take the last element of the lists above
    final_rewards_ucb = [all_cum_rewards_ucb[mc][-1] for mc in range(MC_RUNS)]
    final_rewards_ete = [all_cum_rewards_ete[mc][-1] for mc in range(MC_RUNS)]
    final_rewards_mw = [all_cum_rewards_mw[mc][-1] for mc in range(MC_RUNS)]

    print("\nFINAL CUMULATIVE REWARD (mean ± std)")
    print(f"UCB : {np.mean(final_rewards_ucb):.3f} ± {np.std(final_rewards_ucb):.3f}")
    print(f"EtE : {np.mean(final_rewards_ete):.3f} ± {np.std(final_rewards_ete):.3f}")
    print(f"MW  : {np.mean(final_rewards_mw):.3f} ± {np.std(final_rewards_mw):.3f}")

    plot_arm_selection_per_phase(all_chosen_arms_ucb, all_chosen_arms_mw,"UCB", "MW", "Arm_Selection_Per_Phase_UCB_MW.png")
    plot_arm_selection_per_phase(all_chosen_arms_ete, all_chosen_arms_mw,"EtE", "MW", "Arm_Selection_Per_Phase_EtE_MW.png")
    optimal_cum_reward = optimal_expected_cum_reward()
    plot_comparison(mean_cum_rewards_ucb, mean_cum_rewards_ete, mean_cum_rewards_mw, optimal_cum_reward)


FINAL CUMULATIVE REWARD (mean ± std)
UCB : 11911.550 ± 72.014
EtE : 1839.050 ± 13.970
MW  : 12243.750 ± 128.170


#Part B


## UCB vs Competitor

In [37]:
import numpy as np
import matplotlib.pyplot as plt

T = 10000
MC_RUNS = 10
K = 5
beta = 4.0

np.random.seed(46)
prices = sorted(np.random.uniform(0, 5, K))
print(prices)


def h(q, p_t, beta):
    return 1 + np.exp(beta * (q - p_t)) - beta * q * np.exp(beta * (q - p_t))

def bisection_method(p_t, beta):
    left = 0.0
    right = max(1.0, p_t + 1.0)

    while h(right, p_t, beta) > 0:
        right = 2.0 * right

    while right - left > 1e-8:
      mid = (left + right) / 2.0
      if h(mid, p_t, beta) > 0:
          left = mid
      else:
          right = mid

    return (left + right) / 2.0

def moving_average(x, window):
    smoothed = []
    for i in range(len(x)):
        start = max(0, i - window + 1)
        smoothed.append(np.mean(x[start:i+1]))
    return smoothed


def ucb(mc):

    cum_rewards_ucb = [0.0] * T
    cum_rewards_comp = [0.0] * T
    N = [0] * K
    mu_hat = [0.0] * K
    learner_prices = [0.0] * T
    competitor_prices = [0.0] * T

    cum_reward_ucb = 0.0
    cum_reward_comp = 0.0

    for t in range(T):
        if t < K:
            arm = t
        else:
            ucb_values = []
            for i in range(K):
               # below i set the mutiplier of exploration term equal to the true range of the prices for my seed
                ucb = mu_hat[i] + (3.91 - 1.24) * np.sqrt((2.0 * np.log(T)) / N[i])
                ucb_values.append(ucb)
            arm = ucb_values.index(max(ucb_values))

        p_t = prices[arm]
        q_t = bisection_method(p_t, beta)

        learner_prices[t] = p_t
        competitor_prices[t] = q_t

        prob_learner = 1.0 / (1.0 + np.exp(beta * (p_t - q_t)))
        winner = np.random.binomial(1, prob_learner)

        reward_ucb = p_t * winner
        reward_comp = q_t * (1 - winner)

        cum_reward_ucb += reward_ucb
        cum_reward_comp += reward_comp

        cum_rewards_ucb[t] = cum_reward_ucb
        cum_rewards_comp[t] = cum_reward_comp

        N[arm] += 1
        mu_hat[arm] = mu_hat[arm] + (reward_ucb - mu_hat[arm]) / N[arm]

    return cum_rewards_ucb, cum_rewards_comp,learner_prices, competitor_prices


if __name__ == "__main__":

    all_cum_rewards_ucb = []
    all_cum_rewards_comp = []
    all_learner_prices = []
    all_competitor_prices = []
    final_rewards_ucb = []
    final_rewards_comp = []

    for mc in range(MC_RUNS):
        cum_rewards_ucb, cum_rewards_comp, learner_prices, competitor_prices = ucb(mc)

        all_cum_rewards_ucb.append(cum_rewards_ucb)
        all_cum_rewards_comp.append(cum_rewards_comp)
        all_learner_prices.append(learner_prices)
        all_competitor_prices.append(competitor_prices)

        final_rewards_ucb.append(cum_rewards_ucb[-1])
        final_rewards_comp.append(cum_rewards_comp[-1])

    all_cum_rewards_ucb = np.array(all_cum_rewards_ucb)
    all_cum_rewards_comp = np.array(all_cum_rewards_comp)
    all_learner_prices = np.array(all_learner_prices)
    all_competitor_prices = np.array(all_competitor_prices)

    mean_cum_rewards_ucb = np.mean(all_cum_rewards_ucb, axis=0)
    mean_cum_rewards_comp = np.mean(all_cum_rewards_comp, axis=0)

    mean_learner_prices = np.mean(all_learner_prices, axis=0)
    mean_competitor_prices = np.mean(all_competitor_prices, axis=0)

    mean_final_ucb = np.mean(final_rewards_ucb)
    std_final_ucb = np.std(final_rewards_ucb)

    mean_final_comp = np.mean(final_rewards_comp)
    std_final_comp = np.std(final_rewards_comp)

    print(f"Final cumulative reward Learner: {mean_final_ucb:.3f} ± {std_final_ucb:.3f}")
    print(f"Final cumulative reward Competitor: {mean_final_comp:.3f} ± {std_final_comp:.3f}")

    plt.plot(mean_cum_rewards_ucb, label="Learner")
    plt.plot(mean_cum_rewards_comp, label="Competitor")
    plt.xlabel("Round")
    plt.ylabel("Cumulative Reward")
    plt.title("UCB against best-response competitor")
    plt.legend()
    plt.savefig("Cumulative Reward UCB vs competitor.png", dpi=300, bbox_inches='tight')
    plt.close()

    window = 100
    smooth_learner_prices = moving_average(mean_learner_prices, window)
    smooth_competitor_prices = moving_average(mean_competitor_prices, window)

    plt.plot(smooth_learner_prices, label="Learner price")
    plt.plot(smooth_competitor_prices, label="Competitor price")
    plt.xlabel("Round")
    plt.ylabel("Price")
    plt.title(f"Prices over time UCB vs competitor (moving average, window={window})")
    plt.legend()
    plt.savefig("Prices over time UCB vs competitor", dpi=300, bbox_inches='tight')
    plt.close()



[np.float64(1.2452154575787444), np.float64(1.565384678158399), np.float64(3.1741685302636506), np.float64(3.7903793232490886), np.float64(3.919161754028753)]
Final cumulative reward Learner: 3049.859 ± 66.855
Final cumulative reward Competitor: 19188.271 ± 621.691


## MW vs Competitor

In [38]:
import numpy as np
import matplotlib.pyplot as plt

T = 10000
MC_RUNS = 10
K = 5
beta = 4.0
eta = ((K * np.log(K)) / T) ** (1/3)
epsilon = ((K * np.log(K)) / T) ** (1/3)

np.random.seed(46)
prices = sorted(np.random.uniform(0, 5, K))
print(prices)

def moving_average(x, window):
    smoothed = []
    for i in range(len(x)):
        start = max(0, i - window + 1)
        smoothed.append(np.mean(x[start:i+1]))
    return smoothed

def competitor_expected_reward_mw(q_t, learner_probs, prices):
    value = 0.0
    for i in range(len(prices)):
        p_i = prices[i]
        prob_competitor = 1.0 / (1.0 + np.exp(beta * (q_t - p_i)))
        value += learner_probs[i] * q_t * prob_competitor
    return value

# first derivative of the expected reward
def first_derivative(q_t, learner_probs, prices):
    value = 0.0
    # Because the total derivative is a sum over all learner prices, we start from 0 and keep adding each term
    for i in range(len(prices)):
        p_i = prices[i]
        numerator = 1.0 + (np.exp(beta * (q_t - p_i))) * (1.0 - beta * q_t)
        denumerator = (1.0 + (np.exp(beta * (q_t - p_i)))) ** 2
        value += learner_probs[i] * (numerator / denumerator)
    return value

def bisection_method(learner_probs, prices, tol=1e-8, max_iter=100):
    left = 0.0
    right = max(1.0, max(prices))

    # we enlarge the interval until the derivative becomes negative
    while first_derivative(right, learner_probs, prices) > 0:
        right = 2.0 * right

    for _ in range(max_iter):
        mid = (left + right) / 2.0
        fmid = first_derivative(mid, learner_probs, prices)

        if abs(fmid) < tol:
            return mid

        if fmid > 0:
            left = mid
        else:
            right = mid

    return (left + right) / 2.0


def mw(mc):
    rng = np.random.default_rng(mc)
    # in the beggining we start with all arms having the same weight , each time an arm is played then its weight is being renewed
    weights = np.ones(K)

    cum_rewards_mw = [0.0] * T
    cum_rewards_comp = [0.0] * T
    learner_prices = [0.0] * T
    competitor_prices = [0.0] * T

    cum_reward_mw = 0.0
    cum_reward_comp = 0.0

    for t in range(T):
        p = weights / np.sum(weights)  # we convert the weights to the probability distribution over the arms
        q_probs = (1 - epsilon) * p + epsilon / K  # the arm that will be played is choosen according to this distibution that contains the exploration term

        q_t = bisection_method(q_probs, prices)

        arm = rng.choice(K, p=q_probs)  # the learner chooses an arm probabilistically based on the q_probs distribution
        p_t = prices[arm]

        learner_prices[t] = p_t
        competitor_prices[t] = q_t

        prob_learner = 1.0 / (1.0 + np.exp(beta * (p_t - q_t)))
        # we simulate the customer’s choice ,if winner=1 the learner gets the customer and if winner=0 the competitor gets the customer
        winner = rng.binomial(1, prob_learner)

        reward_mw = p_t * winner
        reward_comp = q_t * (1 - winner)

        cum_reward_mw += reward_mw
        cum_reward_comp += reward_comp

        cum_rewards_mw[t] = cum_reward_mw
        cum_rewards_comp[t] = cum_reward_comp

        # if learner wins then reward = p_t so we have li= 1-normalized_reward else reward =0 and li=1
        li = 1.0 - reward_mw / 3.9
        li_hat = li / q_probs[arm]
        weights[arm] = weights[arm] * ((1 - eta) ** li_hat)

    return cum_rewards_mw, cum_rewards_comp, learner_prices, competitor_prices

if __name__ == "__main__":

    all_cum_rewards_mw = []
    all_cum_rewards_comp = []
    all_learner_prices = []
    all_competitor_prices = []
    final_rewards_mw = []
    final_rewards_comp = []

    for mc in range(MC_RUNS):
        cum_rewards_mw, cum_rewards_comp, learner_prices, competitor_prices = mw(mc)

        all_cum_rewards_mw.append(cum_rewards_mw)
        all_cum_rewards_comp.append(cum_rewards_comp)
        all_learner_prices.append(learner_prices)
        all_competitor_prices.append(competitor_prices)

        final_rewards_mw.append(cum_rewards_mw[-1])
        final_rewards_comp.append(cum_rewards_comp[-1])

    all_cum_rewards_mw = np.array(all_cum_rewards_mw)
    all_cum_rewards_comp = np.array(all_cum_rewards_comp)
    all_learner_prices = np.array(all_learner_prices)
    all_competitor_prices = np.array(all_competitor_prices)

    mean_cum_rewards_mw = np.mean(all_cum_rewards_mw, axis=0)
    mean_cum_rewards_comp = np.mean(all_cum_rewards_comp, axis=0)

    mean_learner_prices = np.mean(all_learner_prices, axis=0)
    mean_competitor_prices = np.mean(all_competitor_prices, axis=0)

    mean_final_mw = np.mean(final_rewards_mw)
    std_final_mw = np.std(final_rewards_mw)

    mean_final_comp = np.mean(final_rewards_comp)
    std_final_comp = np.std(final_rewards_comp)

    print(f"Final cumulative reward Learner: {mean_final_mw:.3f} ± {std_final_mw:.3f}")
    print(f"Final cumulative reward Competitor: {mean_final_comp:.3f} ± {std_final_comp:.3f}")

    plt.plot(mean_cum_rewards_mw, label="Learner")
    plt.plot(mean_cum_rewards_comp, label="Competitor")
    plt.xlabel("Round")
    plt.ylabel("Cumulative Reward")
    plt.title("MW against best-response competitor")
    plt.legend()
    plt.savefig("Cumulative Reward MW vs competitor.png", dpi=300, bbox_inches='tight')
    plt.close()

    window = 100
    smooth_learner_prices = moving_average(mean_learner_prices, window)
    smooth_competitor_prices = moving_average(mean_competitor_prices, window)

    plt.plot(smooth_learner_prices, label="Learner price")
    plt.plot(smooth_competitor_prices, label="Competitor price")
    plt.xlabel("Round")
    plt.ylabel("Price")
    plt.title(f"Prices over time  MW vs competitor (moving average, window={window})")
    plt.legend()
    plt.savefig("Prices over time MW vs competitor.png", dpi=300, bbox_inches='tight')
    plt.close()

[np.float64(1.2452154575787444), np.float64(1.565384678158399), np.float64(3.1741685302636506), np.float64(3.7903793232490886), np.float64(3.919161754028753)]
Final cumulative reward Learner: 3604.559 ± 192.505
Final cumulative reward Competitor: 8281.896 ± 424.811


Plot of learner's winning probability for β = {0.2 , 0.5 , 4.0}

In [ ]:
    x = np.linspace(-2.5, 2.5, 500)

    beta_values = [0.2 ,0.5, 4.0]

    for b in beta_values:
        y = 1.0 / (1.0 + np.exp(b * x))
        plt.plot(x, y, label=fr'$\beta={b}$')

    plt.axvline(0.0, linewidth=1.0)
    plt.axhline(0.5, linewidth=1.0)

    plt.xlabel(r'Price gap $p_t - q_t$')
    plt.ylabel(r'$P(\mathrm{learner\ wins})$')
    plt.title('Learner-winning probability for different values of $\\beta$')
    plt.legend()
    plt.savefig("learner_winning_probability_curve.png", dpi=300, bbox_inches='tight')
    plt.close()